# HeartTwin: KIT and EDGAR real-data benchmarks
Pinned source: `351aaa776f414087afb0fd49575080805e34ff81`.
Runs on a standard CPU Colab runtime. Raw downloads come only from the dataset owners, with pinned checksums. No hospital data or credentials are used.

KIT: original anatomy/data audit plus an explicitly assumed native 0D circulation fit (23 calibration samples, 12 temporal holdout samples; RV end-systolic volume is a separate chamber target). The split is retrospective after raw-data inspection; this is not preregistered or prospectively blinded. No selection or retuning on holdout.

EDGAR Bratislava P034: audit the actual documentation, surface geometry and electrode mapping. At inspection, the signed-in source page linked no Run1/Run2 waveform archive. Electrical prediction metrics must remain absent unless those missing files are obtained and qualified.

KIT data licence: CC BY-NC 4.0, Gerach et al.; see source receipts. EDGAR attribution and rights are retained in its receipts. This is not clinical validation or a complete spatial whole-heart benchmark.



In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, traceback, zipfile
SOURCE_COMMIT = '351aaa776f414087afb0fd49575080805e34ff81'
CARDIMECH_COMMIT = '5dc14d886ccbda5a1398b863f9ae7821bbffa18b'
ROOT = Path('/content/HeartTwin_KIT_EDGAR_20261020/retained-' + SOURCE_COMMIT[:12])
ROOT.mkdir(parents=True, exist_ok=True)
DATA = ROOT.parent
RESULTS = ROOT/'results'; RESULTS.mkdir(exist_ok=True)
ENV = dict(os.environ, MPLBACKEND='Agg', PYTHONUNBUFFERED='1')
def run(command, name, timeout=900):
    print('Starting', name, flush=True)
    with (RESULTS/(name+'.log')).open('w') as log:
        completed = subprocess.run(list(map(str, command)), stdout=log, stderr=subprocess.STDOUT, env=ENV, timeout=timeout)
    if completed.returncode:
        print((RESULTS/(name+'.log')).read_text()[-10000:])
        raise RuntimeError(name+' failed: '+str(completed.returncode))
repo = ROOT/'source'
if not repo.exists():
    run(['git','init',repo], 'git-init')
    run(['git','-C',repo,'fetch','--depth','1','https://github.com/Virelion-Biotech/Virelion-HeartTwin.git',SOURCE_COMMIT], 'git-fetch')
    run(['git','-C',repo,'checkout','--detach','FETCH_HEAD'], 'git-checkout')
assert subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip()==SOURCE_COMMIT
assert not subprocess.check_output(['git','-C',str(repo),'status','--porcelain','--untracked-files=no'],text=True).strip()
venv = ROOT/'venv'
if not venv.exists(): run([sys.executable,'-m','venv','--without-pip',venv], 'venv')
python = venv/'bin/python'
ENV['PATH'] = str(python.parent)+os.pathsep+ENV['PATH']
run([sys.executable,'-m','pip','--python',python,'install','pip==25.1.1','numpy==2.2.6','scipy==1.16.3','pydantic==2.12.3','PyYAML==6.0.3','meshio==5.3.5','matplotlib==3.10.7','ruff==0.14.1'], 'install-base')
run([python,'-m','pip','install','-e',repo,'git+https://github.com/Virelion-Biotech/Virelion-CardiMech.git@'+CARDIMECH_COMMIT], 'install-projects')
run([python,'-m','pip','check'], 'pip-check')
(RESULTS/'source_pin.json').write_text(json.dumps({'source_commit':SOURCE_COMMIT,'CardiMech_commit':CARDIMECH_COMMIT,'original_KIT_run_commit':'492a83a2a3b4f9627892f77ff7f958a72ffe511e','execution':'User Edge Colab','scope':'Exploratory KIT circulation plus EDGAR geometry audit; no whole-heart accuracy claim'},indent=2))
state={'status':'STARTED','KIT':'NOT_RUN','EDGAR':'NOT_RUN','clinical_qualification':'NOT_ESTABLISHED'}
try:
    run([python,repo/'scripts/benchmark_kit_edgar.py','--data-dir',DATA,'--output',RESULTS], 'KIT-benchmark', timeout=600)
    state['KIT']='COMPLETED_EXPLORATORY'
    run([python,repo/'scripts/benchmark_edgar_p034.py','--data-dir',DATA/'edgar','--output',RESULTS], 'EDGAR-audit', timeout=300)
    state['EDGAR']=json.loads((RESULTS/'EDGAR_status.json').read_text())['status']
    run([python,'-m','py_compile',repo/'scripts/benchmark_kit_edgar.py',repo/'scripts/benchmark_edgar_p034.py'], 'compile')
    run([python,'-m','ruff','check',repo/'scripts/benchmark_kit_edgar.py',repo/'scripts/benchmark_edgar_p034.py'], 'lint')
    state['status']='EXPLORATORY_RUN_COMPLETE'
except BaseException:
    state.update(status='FAILED',error=traceback.format_exc())
    print(state['error'])
finally:
    (RESULTS/'RUN_STATUS.json').write_text(json.dumps(state,indent=2))
print(json.dumps(state,indent=2))
if state['status']=='FAILED': raise RuntimeError('Retain failure logs; do not report a pass.')



In [ ]:
PLOT_SCRIPT = "from pathlib import Path\nimport json\nimport numpy as np\nimport matplotlib\nmatplotlib.use('Agg')\nimport matplotlib.pyplot as plt\nimport sys\nroot=Path(sys.argv[1])\nr=json.loads((root/'KIT_circulation_results.json').read_text())\nprediction=json.loads((root/'frozen_prediction.json').read_text())\nrows=np.genfromtxt(root/'LV_predictions.csv',delimiter=',',names=True,dtype=None,encoding='utf-8')\nt=rows['time_s']; observed=rows['observed_LV_mL']; model=rows['predicted_LV_mL']\nfig,axes=plt.subplots(2,1,figsize=(9,8),constrained_layout=True)\naxes[0].plot(t,observed,'o-',color='#164e63',label='Measured LV volume',markersize=4)\naxes[0].plot(t,model,'-',color='#b45309',label='Conditional native 0D model')\naxes[0].plot(t[23:],[observed[22]]*12,'--',color='#64748b',label='Persistence baseline')\naxes[0].axvspan((t[22]+t[23])/2,t[-1],color='#fef3c7',label='Temporal holdout')\naxes[0].set(xlabel='Time (s)',ylabel='LV volume (mL)',title='KIT volunteer: reduced circulation underperforms persistence')\naxes[0].legend(fontsize=8,loc='lower right'); axes[0].grid(alpha=.2)\np=np.array(prediction['pressure_mmHg']['LV'])\nfor scale,color in [(.5,'#0f766e'),(1.,'#64748b'),(2.,'#be123c')]:\n    axes[1].plot(t,p*scale,color=color,label=f'Pressure scale {scale:g}; identical volume trajectory')\naxes[1].set(xlabel='Time (s)',ylabel='Model LV pressure (mmHg)',title='Pressure is not identified by volume-only fitting')\naxes[1].legend(fontsize=8); axes[1].grid(alpha=.2)\nfig.suptitle('Exploratory single-volunteer result; no spatial mechanics or clinical validation',fontsize=11)\nfig.savefig(root/'KIT_results.png',dpi=160)\nfig.savefig(root/'KIT_results.svg')\nplt.close(fig)\n(root/'plot_environment.json').write_text(json.dumps({'matplotlib':matplotlib.__version__,'numpy':np.__version__},indent=2))\n"
from pathlib import Path
import hashlib, json, subprocess
assert json.loads((RESULTS/'RUN_STATUS.json').read_text())['status']=='EXPLORATORY_RUN_COMPLETE'
r=json.loads((RESULTS/'KIT_circulation_results.json').read_text())
g=json.loads((RESULTS/'KIT_geometry_audit.json').read_text())
e=json.loads((RESULTS/'EDGAR_status.json').read_text())
assert r['calibration']['n']==23 and r['temporal_holdout']['n']==12
assert hashlib.sha256((RESULTS/'frozen_prediction.json').read_bytes()).hexdigest()==r['frozen_prediction_sha256']
assert g['nodes']==32505 and g['tetrahedra']==128976
assert e['electrode_mapping']['electrodes']==128
assert e['electrical_data']['downloaded_waveform_records']==0
plot_script=ROOT/'plot_results.py'
plot_script.write_text(PLOT_SCRIPT)
run([python,plot_script,RESULTS], 'figures')
print('HEARTTWIN — KIT / EDGAR REAL-DATA RESULTS')
print('KIT: LV fit 23 samples | holdout 12 samples')
print('LV holdout RMSE %.2f mL; persistence baseline %.2f mL'%(r['temporal_holdout']['RMSE_mL'],r['persistence_baseline_holdout']['RMSE_mL']))
print('RV withheld volume: absolute error %.2f mL'%r['RV_ES_holdout']['absolute_error_mL'])
print('Pressure scale is NOT IDENTIFIABLE from volumes.')
print('KIT spatial mesh REJECTED: 32,505 nodes / 128,976 tetrahedra.')
print('EDGAR: 7 surface meshes / 128 electrode mappings audited.')
print('EDGAR waveform records: 0; electrical prediction NOT RUN.')
print('Compile / lint / artifact consistency checks completed.')
print('Clinical accuracy NOT ESTABLISHED.')
print('SOURCE',SOURCE_COMMIT)
print('EDGAR_MAPPING',json.dumps(e['electrode_mapping']))
print('EDGAR_TOPOLOGY',json.dumps({k:{n:v[n] for n in ['edges_without_two_faces','inconsistent_directed_edges','duplicate_triangles','degenerate_triangles']} for k,v in e['surfaces'].items()}))



In [ ]:
REPORT_TEXT = "# KIT and EDGAR real-data results — 20 October 2026\n\nThese are two separate human subjects. KIT supports an exploratory volume-fit benchmark and a mesh compatibility audit. EDGAR P034 supports a surface/electrode audit; its inspected download page does not supply the documented waveform recordings. Neither establishes clinical accuracy or a fully personalised whole-heart model.\n\n## What was executed\n\nAll numerical work ran in the user's Edge Colab. Source `351aaa776f414087afb0fd49575080805e34ff81`; CardiMech `5dc14d886ccbda5a1398b863f9ae7821bbffa18b`. The initial KIT run used `492a83a2a3b4f9627892f77ff7f958a72ffe511e`. The later source removes the obsolete initial EDGAR login status and adds the independent EDGAR audit; the KIT fitting method, bounds, starting points and holdout partition are unchanged.\n\nThe reusable notebook is [HeartTwin_KIT_EDGAR_Real_Data.ipynb](../notebooks/HeartTwin_KIT_EDGAR_Real_Data.ipynb). Retained logs, numerical outputs, source receipts, assumptions and hashes are in [validation/kit_edgar_2026_10_20](../validation/kit_edgar_2026_10_20/).\n\n## KIT: real measurements, conditional reduced circulation\n\nSource: [Gerach et al., Zenodo 10526554, version 1.1](https://zenodo.org/records/10526554). Publisher MD5 checksums and independent SHA-256 hashes were verified for Data.zip, LabelIDs.txt, M.vtu, Surfaces.zip and TetGen.zip. EP.vtu was not needed for this mechanics/circulation audit. Source licence is CC BY-NC 4.0; source archives are not redistributed here. Derived measurement files retain that attribution and noncommercial restriction.\n\nThe source is one male volunteer, not a disease cohort. There are 35 finite LV endocardial-volume samples, 35 epicardial enclosed-volume samples, and only two RV endocardial-volume samples. Epicardial enclosed volume is not blood-pool volume. AVPD and AVPV traces retain their own time grids; regional wall-thickening data have 17 labelled segments. Missing RV values were not filled. No phase shift, time warping or assumed simultaneous acquisition was introduced.\n\n### Fit and holdout\n\nThe benchmark calls HeartTwin's actual `simulate_closed_loop` implementation: eight closed-loop compartments, elastance chambers, compliant vessels and ideal diode valves. It does not use the 3D spatial mechanics solver.\n\nThe first 23 LV samples are calibration data; the final 12 samples are a temporal holdout. Initial LV and RV volumes are supplied measurements. The second RV volume is excluded from fitting and evaluated as an additional chamber target. Two fixed optimiser starts are compared by calibration SSE only, then predictions are saved and hash-frozen before the scoring code opens the holdout values. No hyperparameters or model assumptions are retuned to improve the holdout result.\n\nThis is a retrospective split after inspection of the complete raw table, not preregistered or prospectively blinded validation. Samples are correlated and belong to one acquisition/volunteer. The standalone research runner does not claim to have passed the stricter PatientCase shared-source evidence gate: calibration and holdout derive from one publisher archive.\n\nOnly three parameters are fitted: LV maximum elastance, LV activation duration, and LA-to-LV resistance. All remaining chamber/vessel parameters, other initial compartment volumes, valve resistances and the 1.3-second cycle length are explicit analyst assumptions, not patient measurements. No burn-in or periodic steady-state condition is asserted. This limited experiment tests one conditional configuration, not all possible personalised calibrations.\n\n| Quantity | Result |\n|---|---:|\n| LV calibration RMSE, 23 samples | 13.72774 mL |\n| LV temporal holdout RMSE, 12 samples | 22.86855 mL |\n| Persistence baseline holdout RMSE | 8.06134 mL |\n| Maximum LV holdout absolute error | 35.76061 mL |\n| Withheld RV volume observed | 65.938406 mL |\n| Withheld RV volume predicted | 87.810397 mL |\n| Withheld RV absolute error | 21.871991 mL |\n| Total-volume conservation drift | 1.82e-12 mL |\n| Maximum LV change with tighter integration tolerances | 0.000626 mL |\n\nThe reduced model is worse than persistence on the declared temporal holdout. Both optimiser starts settle near the same fitted values, with mitral-edge resistance at its imposed lower bound (0.001 mmHg s/mL). Optimiser termination is not proof of a uniquely identified or physiologically correct fit.\n\n### Pressure nonidentifiability\n\nMultiplying all chamber elastances, external pressures and edge resistances by the same positive factor, while dividing vessel compliances by that factor, leaves volume dynamics unchanged. The numerical checks at factors 0.5 and 2 reproduce exactly the same volume trajectories at stored precision, despite LV peak pressures of approximately 76.07 and 304.29 mmHg. These are alternative model outputs, not patient pressure measurements.\n\nVolume-only fitting therefore cannot establish absolute pressure, elastance and resistance scales in this model. A tighter integration result and excellent volume conservation establish numerical consistency only; they do not fix physiological nonidentifiability. No confidence intervals, posterior uncertainty or clinical acceptance threshold are claimed.\n\n### Spatial mechanics compatibility\n\nThe supplied official TetGen mesh contains 32,505 nodes and 128,976 tetrahedra; all tetrahedral determinants are positive. The current CardiMech loader rejects the size preflight with `Mesh requires 4..5000 finite 3D nodes`; its element ceiling is also 20,000. No limits were disabled.\n\nAll four supplied cavity STL surfaces have two faces per edge and consistent local orientation. Their geometric enclosed volumes are approximately LA 70.409, LV 200.563, RA 98.548 and RV 188.512 mL. This does not establish cavity-to-wall registration or expert qualification. The LV mesh cavity volume differs from the first cine volume (221.645 mL); the mesh is derived at diastasis and must not be silently treated as an unloaded end-diastolic reference.\n\nThe rounded TetGen fibre/sheet vectors have a maximum normalised absolute dot product of approximately 1.35e-5, above the current mechanics contract's 1e-10 orthogonality tolerance. Some sheets are not unit length. This is a conversion/qualification requirement, not proof the publisher's model is wrong. No vectors were silently repaired. `meshio 5.3.5` failed to read M.vtu's compressed encoding; the accompanying official TetGen export was audited instead.\n\nA full coupled run still needs a scalable, reviewed mesh conversion; chamber/material/boundary mappings; qualified fibre/sheet handling; constitutive parameters; a defensible loaded reference state; circulation initial conditions; and an activation/voltage model. Spatial displacement, AVPD, regional thickening, stresses and ECG were not predicted. No mesh-convergence claim is made.\n\n## EDGAR Bratislava P034\n\nSource: [signed-in P034 dataset page](https://edgar.sci.utah.edu/2025/07/10/bratislava_2020_p034/). The page was inspected in the user's authenticated Edge session. Its two download links, Documentation-7.zip and Geometries-7.zip, were downloaded and SHA-256 pinned. The archives contain seven surface MAT files, electrode positions, interpolation mappings, a numeric pacing-site position and supporting documentation/images.\n\nThe documented recordings are 128 torso channels plus limb signals R/L/F, 1 kHz, mV, referenced to CMS. The README explicitly distinguishes CMS from Wilson's central terminal. Run1 is RV-apex pacing at 100 bpm; Run2 contains spontaneous PVCs. These facts describe recordings that were not present in either downloaded archive.\n\nThe page lists no waveform download. The audited MAT files contain only surface `pts` and `fac`, not `potvals`. No unlisted URL was guessed, and no simulated waveform was substituted. Consequently ECG error, beat generalisation and pacing-site localisation accuracy remain **not run**, with absent metrics rather than invented zeros.\n\nThe audit preserves explicitly documented zero-based face and triangle indices. It checks edge incidence/orientation, duplicate and degenerate faces, 128 numbered electrode positions, and the supplied interpolation operator. The publisher's mapping illustration uses the second triangle vertex as origin: `position = p2 + a*(p1-p2) + b*(p3-p2)`. This gives weights `[a, 1-a-b, b]` in the published vertex order; no permutation or registration is fitted.\n\nThe electrode coordinate residual is RMS 0.00112576 and maximum 0.00323834 native coordinate units. All 128 rows have nonnegative barycentric weights; the maximum constant-field preservation error is 1.11e-16. This checks the supplied interpolation operator, not a physiological forward model.\n\n| Surface | Edges without exactly two faces | Inconsistent directed edges | Zero-area triangles |\n|---|---:|---:|---:|\n| EndoEpi | 6 | 12 | 8 |\n| Epi | 0 | 0 | 2 |\n| Right lung | 2 | 4 | 0 |\n| Left lung | 1 | 2 | 0 |\n| Torso | 0 | 0 | 0 |\n| RV_RA | 3 | 6 | 0 |\n| LV_LA | 0 | 0 | 0 |\n\nNo duplicate indexed triangles were found. The incidence/orientation counts can describe related defects and must not be added as independent defect totals. The audit does not establish absence of self-intersections, correct nesting, image registration or source accuracy. These surfaces need qualification before assuming a valid conductor domain; no repair was applied.\n\nQuantitative topology and coordinate-residual results are retained in EDGAR_status.json. Coordinate units are not explicitly declared in the inspected P034 MAT/readme metadata, so residuals and volumes remain in native units. No metre/mm conversion, standard V1–V6 montage, or channel alignment is invented.\n\nThese surfaces do not directly supply the tetrahedral torso/myocardium conductivity model required by HeartTwin's reciprocal FEM lead-field implementation. The readme identifies the pacing position from CT; the broader catalogue's MRI description is insufficient to establish the modality of every supplied geometry. Run2 ablation reference is verbal plus a CARTO image, not a precise numerical target. No cine mechanics or chamber pressures were found in the downloaded files.\n\nEDGAR attribution: Institute of Measurement Science, Slovak Academy of Sciences and National Institute for Cardiovascular Diseases, Bratislava; Aras et al. 2015, doi:10.1016/j.jelectrocard.2015.08.008; Svehlikova et al., CinC 2019, doi:10.22489/CinC.2019.303, and the additional references on the source page/readme. Source archives/images/meshes are not redistributed; no explicit licence identifier was supplied in the inspected materials.\n\n## What resolves the remaining gaps\n\n1. **KIT:** improve the model only against calibration data or a new prospectively defined experiment; preserve this failed baseline. Qualify spatial conversion, reference state and boundary conditions before claiming mechanics validation. Add pressure/flow measurements or constrained independent evidence to address scale ambiguity.\n2. **EDGAR P034:** obtain Run1/Run2 waveform arrays and their montage/time/reference metadata from the maintainers. Confirm geometry units and conductivity assumptions, then freeze a beat/site/channel split before fitting. A screenshot of a CARTO ablation is insufficient for a numeric location-error claim.\n3. **Both:** use new subjects/conditions for independent validation and uncertainty assessment. Do not combine these two people's measurements into one patient.\n\nNo hospital integration, trained segmentation model, clinical benefit or whole-heart predictive accuracy was validated by this work.\n"
import base64, hashlib, json, zipfile
(RESULTS/'RESULTS_AND_LIMITATIONS.md').write_text(REPORT_TEXT)
(RESULTS/'plot_results.py').write_text(PLOT_SCRIPT)
selected=[p for p in RESULTS.rglob('*') if p.is_file() and p.suffix.lower() not in {'.png'} and p.name not in {'ARTIFACT_HASHES.json','export_bundle.json'}]
hashes={str(p.relative_to(RESULTS)):hashlib.sha256(p.read_bytes()).hexdigest() for p in selected}
(RESULTS/'ARTIFACT_HASHES.json').write_text(json.dumps(hashes,indent=2))
archive=ROOT/'HeartTwin_KIT_EDGAR_Results.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in selected+[RESULTS/'ARTIFACT_HASHES.json']:
        z.write(p,str(p.relative_to(RESULTS)))
bundle={'archive_sha256':hashlib.sha256(archive.read_bytes()).hexdigest(),'archive_bytes':archive.stat().st_size,'archive_base64':base64.b64encode(archive.read_bytes()).decode(),'text_files':{str(p.relative_to(RESULTS)):p.read_text() for p in selected+[RESULTS/'ARTIFACT_HASHES.json']}}
(RESULTS/'export_bundle.json').write_text(json.dumps(bundle))
print('BUNDLE_SIZE',len(json.dumps(bundle)))
print('ARCHIVE_SHA256',bundle['archive_sha256'])
print('ARCHIVE_BYTES',bundle['archive_bytes'])
print('FILES',list(bundle['text_files']))



In [ ]:
from google.colab import files
files.download(str(archive))
